# 01 — Single Run & Dashboard

Run the heterogeneous-agent market ABM with default parameters and visualize the simulation dashboard.

In [ ]:
import sys
import os
sys.path.insert(0, '..')
os.makedirs('../figures', exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
from market_abm import MarketModel, DEFAULT_PARAMS
from market_abm.analytics import compute_return_statistics, validate_stylized_facts
from market_abm.visualization import (
    plot_price_and_fundamental, plot_spread_over_time,
    plot_volume_over_time, plot_wealth_evolution,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## Run the model

In [ ]:
params = {
    **DEFAULT_PARAMS,
    'steps': 3000,
    'n_agents': 200,
    'seed': 42,
}

model = MarketModel(params)
results = model.run()
print('Simulation complete.')

## Extract data

In [ ]:
data = results.variables.MarketModel
returns = data['log_return'].values

print(f'Price range: [{data["price"].min():.2f}, {data["price"].max():.2f}]')
print(f'Number of steps: {len(returns)}')
print()

# Summary statistics
stats = compute_return_statistics(returns)
for k, v in stats.items():
    print(f'{k:>15s}: {v}')

## Overview plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_price_and_fundamental(data, ax=axes[0, 0])
plot_wealth_evolution(data, ax=axes[0, 1])
plot_spread_over_time(data, ax=axes[1, 0])
plot_volume_over_time(data, ax=axes[1, 1])
plt.tight_layout()
fig.savefig('../figures/dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## Stylized Facts Validation

In [ ]:
facts = validate_stylized_facts(returns)
for name, info in facts.items():
    status = '✅' if info['passed'] else '❌'
    print(f'{status} {name}')
    for k, v in info.items():
        if k != 'passed':
            print(f'    {k}: {v}')
    print()

## Reporter Summary

In [ ]:
results.reporters